[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IyadSultan/CCI/blob/main/session3/student/Assignment_Pharmacy_Pipeline_Student.ipynb)

# Session 3 Assignment: The Complete Pharmacy Intelligence Pipeline
## CCI — Clinical AI Development

**Due:** Before Session 4  
**Estimated effort:** 4-6 hours  
**Submission:** Push to your `cci-course-notebooks` GitHub repo under `assignments/session-3/`

### Clinical Scenario
> KHCC's pharmacy department processes hundreds of medication orders daily through the VistA system. Pharmacists need to verify orders, check for drug interactions, and answer clinical questions about prescribing patterns. Currently this requires manual SQL queries and cross-referencing paper notes. You will build an end-to-end pharmacy intelligence pipeline that loads medication data into SQL, extracts drug interactions from clinical notes using structured AI output, and provides a natural language query interface for pharmacists.

### What You Will Build
This assignment ties together **all five Session 3 labs**:
- **Lab 1** (API Fundamentals) — API calls, token awareness
- **Lab 2** (Structured Output) — Pydantic extraction from clinical notes
- **Lab 3** (Tool Calling) — stretch challenge
- **Lab 4** (CSV to SQL) — loading pharmacy data, writing queries
- **Lab 5** (Text-to-SQL) — natural language pharmacy queries

### Grading
| Part | Weight | Description |
|------|--------|-------------|
| Part 1 | 25% | CSV to SQL — Load and query pharmacy data |
| Part 2 | 25% | Structured Output — Extract drug info from clinical notes |
| Part 3 | 25% | Text-to-SQL — Pharmacist natural language interface |
| Part 4 | 15% | Critical Analysis — Clinical risks and limitations |
| Part 5 | 10% | Stretch Challenge — Tool calling OR benchmarking |

### Anti-Shortcut Rules
- You MUST use the provided data files (`vista_pharmacy.csv`, `vista_patients.csv`, `synthetic_clinical_notes.json`)
- Your Part 4 analysis must reference YOUR specific implementation
- All outputs must reproduce from a top-to-bottom notebook run
- Push to GitHub with at least 3 separate commits with descriptive messages
- No real patient data in the notebook

---
## Setup

In [71]:
# Setup — clone the repo and install packages
import os

if not os.path.exists('/content/CCI'):
    !git clone https://github.com/IyadSultan/CCI.git /content/CCI
os.chdir('/content/CCI/session3/student')

!pip install -q openai pydantic pandas

import json
import pandas as pd
import sqlite3
from openai import OpenAI
from pydantic import BaseModel
from typing import List, Optional
from google.colab import userdata

# Make sure your API key is stored in Colab Secrets (click the key icon in the left sidebar)
# under the name: OPENAI_API_KEY
api_key = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=api_key)

print(f"Working directory: {os.getcwd()}")
print("Setup complete!")

Working directory: /content/CCI/session3/student
Setup complete!


---
# Part 1: CSV to SQL — Load and Query Pharmacy Data (25%)

Load the VistA pharmacy export and patient demographics into a SQLite database,
explore the data, and write SQL queries to answer clinical pharmacy questions.

**Data files:**
- `../data/vista_pharmacy.csv` — 500 medication orders (122 columns)
- `../data/vista_patients.csv` — 79 patient demographics

**Key pharmacy columns:** `MRN`, `DRUG`, `QTY`, `DAYS_SUPPLY`, `Number_OF_REFILLS`, `ROUTE`, `SCHEDULE`, `STATUS`, `CLINIC`, `PROVIDER`, `PHARMACY_ORDERABLE_ITEM`, `DOSAGE_ORDERED`, `ISSUE_DATE`, `FILL_DATE`, `EXPIRATION_DATE`

**JOIN key:** `MRN` links pharmacy orders to patient demographics.

### Cell 1.1 — Load CSVs into SQLite

In [72]:
# === CELL 1.1: LOAD DATA INTO SQLITE ===

# TODO: Load vista_pharmacy.csv into a DataFrame
df_pharmacy = pd.read_csv('/content/CCI/session3/data/vista_pharmacy.csv')

# TODO: Load vista_patients.csv into a DataFrame
df_patients = pd.read_csv('/content/CCI/session3/data/vista_patients.csv')

# TODO: Create a SQLite connection and load both tables
conn = sqlite3.connect('khcc_pharmacy.db')
df_pharmacy.to_sql('vista_pharmacy', conn, if_exists='replace', index=False)
df_patients.to_sql('vista_patients', conn, if_exists='replace', index=False)

# TODO: Verify both tables loaded correctly
# Print row counts for each table
print(f"vista_pharmacy table loaded with {pd.read_sql('SELECT COUNT(*) FROM vista_pharmacy', conn).iloc[0,0]} rows")
print(f"vista_patients table loaded with {pd.read_sql('SELECT COUNT(*) FROM vista_patients', conn).iloc[0,0]} rows")

vista_pharmacy table loaded with 500 rows
vista_patients table loaded with 79 rows


### Cell 1.2 — Explore Pharmacy Data

In [73]:
# === CELL 1.2: EXPLORE PHARMACY DATA ===

# TODO: Print the shape of the pharmacy DataFrame
print(f"Shape of df_pharmacy: {df_pharmacy.shape}")

# TODO: Show the distribution of DRUG (top 10 most prescribed)
print("\nTop 10 most prescribed DRUGs:")
print(df_pharmacy['DRUG'].value_counts().head(10))

# TODO: Show the distribution of STATUS (ACTIVE, EXPIRED, DISCONTINUED, etc.)
print("\nDistribution of STATUS:")
print(df_pharmacy['STATUS'].value_counts())

# TODO: Show the distribution of ROUTE (ORAL, TOPICAL, SUBCUTANEOUS, etc.)
print("\nDistribution of ROUTE:")
print(df_pharmacy['ROUTE'].value_counts())

# TODO: How many unique patients have pharmacy orders?
unique_patients = df_pharmacy['MRN'].nunique()
print(f"\nNumber of unique patients with pharmacy orders: {unique_patients}")

# TODO: What is the date range of ISSUE_DATE?
# Ensure 'ISSUE_DATE' is in datetime format first for accurate min/max
df_pharmacy['ISSUE_DATE'] = pd.to_datetime(df_pharmacy['ISSUE_DATE'], errors='coerce')
min_issue_date = df_pharmacy['ISSUE_DATE'].min()
max_issue_date = df_pharmacy['ISSUE_DATE'].max()
print(f"\nDate range of ISSUE_DATE: {min_issue_date} to {max_issue_date}")

Shape of df_pharmacy: (500, 122)

Top 10 most prescribed DRUGs:
DRUG
LETROZOLE 2.5MG TAB                      30
ANASTROZOLE 1MG TAB                      28
ESOMEPRAZOLE 40MG TAB                    25
CELECOXIB 200MG CAP                      24
ERLOTINIB 150MG TAB                      23
CALCIUM CARBONATE/VITAMIN D 650MG TAB    23
metFORMIN 500MG TAB                      22
TAMOXIFEN 20MG TAB                       22
IMATINIB 400MG TAB                       22
GABAPENTIN 300MG CAP                     21
Name: count, dtype: int64

Distribution of STATUS:
STATUS
EXPIRED                219
DISCONTINUED           146
ACTIVE                  68
DISCONTINUED (EDIT)     67
Name: count, dtype: int64

Distribution of ROUTE:
ROUTE
ORAL            415
TOPICAL          47
SUBCUTANEOUS     38
Name: count, dtype: int64

Number of unique patients with pharmacy orders: 79

Date range of ISSUE_DATE: 2017-01-10 00:00:00 to 2025-12-16 00:00:00


### Cell 1.3 — SQL Queries (5 required)

Write these 5 SQL queries using `pd.read_sql()`:
1. **Top 10 prescribed drugs** — count of orders per drug, ordered descending
2. **Prescriptions per clinic** — count of orders per CLINIC
3. **Active vs expired** — count by STATUS
4. **Average days supply by drug** — for the top 5 drugs
5. **Patients with refills** — patients who have `Number_OF_REFILLS > 0`, show MRN, DRUG, refill count

In [74]:
# === CELL 1.3: SQL QUERIES ===

# Query 1: Top 10 prescribed drugs
print("\nQuery 1: Top 10 prescribed drugs")
query1 = """SELECT DRUG, COUNT(*) AS order_count FROM vista_pharmacy GROUP BY DRUG ORDER BY order_count DESC LIMIT 10"""
print(pd.read_sql(query1, conn))

# Query 2: Prescriptions per clinic
print("\nQuery 2: Prescriptions per clinic")
query2 = """SELECT CLINIC, COUNT(*) AS prescription_count FROM vista_pharmacy GROUP BY CLINIC ORDER BY prescription_count DESC"""
print(pd.read_sql(query2, conn))

# Query 3: Count by STATUS
print("\nQuery 3: Count by STATUS")
query3 = """SELECT STATUS, COUNT(*) AS status_count FROM vista_pharmacy GROUP BY STATUS"""
print(pd.read_sql(query3, conn))

# Query 4: Average days supply for top 5 drugs
print("\nQuery 4: Average days supply by drug (top 5 most prescribed)")
query4 = """SELECT DRUG, AVG(DAYS_SUPPLY) AS avg_days_supply FROM vista_pharmacy WHERE DRUG IN (SELECT DRUG FROM vista_pharmacy GROUP BY DRUG ORDER BY COUNT(*) DESC LIMIT 5) GROUP BY DRUG ORDER BY avg_days_supply DESC"""
print(pd.read_sql(query4, conn))

# Query 5: Patients with refills > 0
print("\nQuery 5: Patients with refills > 0")
query5 = """SELECT MRN, DRUG, Number_OF_REFILLS FROM vista_pharmacy WHERE Number_OF_REFILLS > 0"""
print(pd.read_sql(query5, conn))


Query 1: Top 10 prescribed drugs
                                    DRUG  order_count
0                    LETROZOLE 2.5MG TAB           30
1                    ANASTROZOLE 1MG TAB           28
2                  ESOMEPRAZOLE 40MG TAB           25
3                    CELECOXIB 200MG CAP           24
4                    ERLOTINIB 150MG TAB           23
5  CALCIUM CARBONATE/VITAMIN D 650MG TAB           23
6                    metFORMIN 500MG TAB           22
7                     TAMOXIFEN 20MG TAB           22
8                     IMATINIB 400MG TAB           22
9                   GABAPENTIN 300MG CAP           21

Query 2: Prescriptions per clinic
                      CLINIC  prescription_count
0     PULMONARY-FERAS HAWARI                  56
1      HEMATOLOGY-LINA BSOUL                  46
2        PEDIATRIC ONC-AHMAD                  45
3       GI ONC-ISSA TARAWNEH                  44
4             LUNG ONC-FERAS                  43
5            KHCC-3C-SURGERY               

### Cell 1.4 — JOIN Queries (3 required)

Combine pharmacy orders with patient demographics:
1. **Prescriptions by nationality** — JOIN on MRN, group by NATIONALITY
2. **Drugs for patients over 60** — calculate age from DOB, find their medications
3. **Male vs female prescription counts** — JOIN on MRN, group by SEX

In [75]:
# === CELL 1.4: JOIN QUERIES ===

# JOIN Query 1: Prescriptions by nationality
print("\nJOIN Query 1: Prescriptions by nationality")
query_join1 = """SELECT tp.NATIONALITY, COUNT(vp.MRN) AS prescription_count FROM vista_pharmacy vp JOIN vista_patients tp ON vp.MRN = tp.MRN GROUP BY tp.NATIONALITY ORDER BY prescription_count DESC"""
print(pd.read_sql(query_join1, conn))

# JOIN Query 2: Drugs for patients over 60 (calculate age from DOB)
# Hint: You can use julianday('now') - julianday(DOB) in SQLite
print("\nJOIN Query 2: Drugs for patients over 60")
query_join2 = """SELECT DISTINCT vp.DRUG FROM vista_pharmacy vp JOIN vista_patients tp ON vp.MRN = tp.MRN WHERE (julianday('now') - julianday(tp.DOB)) / 365.25 > 60"""
print(pd.read_sql(query_join2, conn))

# JOIN Query 3: Male vs female prescription counts
print("\nJOIN Query 3: Male vs female prescription counts")
query_join3 = """SELECT tp.SEX, COUNT(vp.MRN) AS prescription_count FROM vista_pharmacy vp JOIN vista_patients tp ON vp.MRN = tp.MRN GROUP BY tp.SEX"""
print(pd.read_sql(query_join3, conn))


JOIN Query 1: Prescriptions by nationality
  NATIONALITY  prescription_count
0         JOR                 312
1         PSE                  79
2         SYR                  57
3         IRQ                  24
4         SAU                  18
5         LBN                   5
6         EGY                   5

JOIN Query 2: Drugs for patients over 60
                                     DRUG
0                     ENOXAPARIN 40MG INJ
1          FUSIDIC ACID 20MG/GM CREAM 15G
2   CALCIUM CARBONATE/VITAMIN D 650MG TAB
3                   DOXYCYCLINE 100MG CAP
4                   FILGRASTIM 300MCG INJ
5                      SUNITINIB 50MG CAP
6                      TAMOXIFEN 20MG TAB
7                   ESOMEPRAZOLE 40MG TAB
8                    GABAPENTIN 300MG CAP
9               MORPHINE SULFATE 10MG TAB
10                    CELECOXIB 200MG CAP
11                    LETROZOLE 2.5MG TAB
12            CEFUROXIME AXETIL 500MG TAB
13                     IMATINIB 400MG TAB
14          

---
# Part 2: Structured Output — Extract Drug Info from Clinical Notes (25%)

Use Pydantic structured output to extract medications, allergies, and potential drug interactions
from 10 synthetic oncology notes. Then cross-reference extracted medications against the
pharmacy database.

**Data file:** `../data/synthetic_clinical_notes.json`

### Cell 2.1 — Define Pydantic Models

In [76]:
# === CELL 2.1: PYDANTIC MODELS ===

# TODO: Define a DrugMention model with:
#   name (str), dose (Optional[str]), frequency (Optional[str]), route (Optional[str])
class DrugMention(BaseModel):
    name: str
    dose: Optional[str]
    frequency: Optional[str]
    route: Optional[str]

# TODO: Define a DrugInteractionAlert model with:
#   drug1 (str), drug2 (str), risk_level (str: "high", "moderate", "low"),
#   description (str)
class DrugInteractionAlert(BaseModel):
    drug1: str
    drug2: str
    risk_level: str  # "high", "moderate", "low"
    description: str

# TODO: Define a NoteExtraction model with:
#   patient_name (str), mrn (str), medications (List[DrugMention]),
#   allergies (List[str]), potential_interactions (List[DrugInteractionAlert]),
#   key_findings (List[str])
class NoteExtraction(BaseModel):
    patient_name: str
    mrn: str
    medications: List[DrugMention]
    allergies: List[str]
    potential_interactions: List[DrugInteractionAlert]
    key_findings: List[str]

### Cell 2.2 — Load Clinical Notes

In [77]:
# === CELL 2.2: LOAD CLINICAL NOTES ===

# TODO: Load the 10 notes from the data folder
with open('/content/CCI/session3/data/synthetic_clinical_notes.json', 'r') as f:
    notes = json.load(f)

# TODO: Print how many notes loaded
print(f"Loaded {len(notes)} clinical notes.")

# TODO: Print the first note's ID, type, and first 200 characters of text
if notes:
    first_note = notes[0]
    print(f"First note ID: {first_note['note_id']}")
    print(f"First note type: {first_note['note_type']}")
    print(f"First 200 chars of text: {first_note['text'][:200]}...")

Loaded 10 clinical notes.
First note ID: NOTE_001
First note type: Consultation Note
First 200 chars of text: Patient: Layla Al-Masri, MRN: KH-20241087. Date: 2024-09-15. Referred by Dr. Faris Haddad, Department of General Surgery, KHCC. This 52-year-old female presents for oncology consultation following lef...


### Cell 2.3 — Extract Structured Data from All 10 Notes

Use `client.beta.chat.completions.parse()` with your `NoteExtraction` Pydantic model
to extract structured medication and interaction data from each note.

In [78]:
# === CELL 2.3: EXTRACT FROM ALL 10 NOTES ===

# TODO: Loop through all 10 notes
# For each note, call client.beta.chat.completions.parse() with:
#   model="gpt-4o-mini"
#   response_format=NoteExtraction
#   system prompt: instruct the model to extract medications, allergies,
#                  potential drug interactions, and key findings
#   user message: the note text

# TODO: Store all extractions in a list
# TODO: For each extraction, print:
#   - Note ID and patient name
#   - Number of medications found
#   - Number of potential interactions flagged
#   - Allergies

# === CELL 2.3: EXTRACT FROM ALL 10 NOTES ===
# === CELL 2.3: EXTRACT FROM ALL 10 NOTES ===

extractions = []

for i, note_data in enumerate(notes):
    # note_data["text"] is the raw text we send to the model
    note_text = note_data.get("text", "")

    # Call the OpenAI API with structured output parsing
    completion = client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a an experoeinces clinical pharmacist analyzing medical notes. "
                    "Extract the patient's name, MRN, medications (with dose, frequency, and route), "
                    "known allergies, potential drug-drug interactions, and key clinical findings."
                )
            },
            {"role": "user", "content": note_text},
        ],
        response_format=NoteExtraction,
    )

    # Access the validated NoteExtraction object
    extracted_data = completion.choices[0].message.parsed
    extractions.append(extracted_data)

    # Print summary using the AI-extracted fields
    # This solves the "Unknown" issue by pulling the name directly from the note text
    print(f"--- Note ID: {i} | MRN: {extracted_data.mrn} ---")
    print(f"Patient Name: {extracted_data.patient_name}")
    print(f"Medications Found: {len(extracted_data.medications)}")
    print(f"Potential Interactions: {len(extracted_data.potential_interactions)}")
    print(f"Allergies: {', '.join(extracted_data.allergies) if extracted_data.allergies else 'None'}")

    # Optional: Print a brief snippet of key findings
    if extracted_data.key_findings:
        print(f"Primary Finding: {extracted_data.key_findings[0]}")

    print("-" * 40)

--- Note ID: 0 | MRN: KH-20241087 ---
Patient Name: Layla Al-Masri
Medications Found: 4
Potential Interactions: 0
Allergies: None
Primary Finding: 52-year-old female
----------------------------------------
--- Note ID: 1 | MRN: KH-20230654 ---
Patient Name: Omar Khalil Zarqa
Medications Found: 3
Potential Interactions: 0
Allergies: Penicillin (rash), Sulfa drugs (anaphylaxis)
Primary Finding: Adenocarcinoma of the sigmoid colon, moderately differentiated
----------------------------------------
--- Note ID: 2 | MRN: KH-20240291 ---
Patient Name: Hana Bisharat
Medications Found: 0
Potential Interactions: 0
Allergies: None
Primary Finding: Specimen: Right upper lobe lobectomy
----------------------------------------
--- Note ID: 3 | MRN: KH-20231445 ---
Patient Name: Tariq Awad
Medications Found: 3
Potential Interactions: 0
Allergies: ibuprofen (GI bleed)
Primary Finding: 67 yo male with AML (FAB M4)
----------------------------------------
--- Note ID: 4 | MRN: KH-20240833 ---
Patient 

### Cell 2.4 — Cross-Reference with Pharmacy Database

For each medication extracted from the clinical notes, check if that drug
(by `PHARMACY_ORDERABLE_ITEM`) appears in the `vista_pharmacy` SQL table.
Report which extracted drugs are in the pharmacy system and which are not.

In [79]:


# === CELL 2.4: CROSS-REFERENCE ===

# TODO: Get the list of unique PHARMACY_ORDERABLE_ITEM values from the database
# pharmacy_drugs = pd.read_sql("SELECT DISTINCT PHARMACY_ORDERABLE_ITEM FROM vista_pharmacy", conn)
pharmacy_drugs = pd.read_sql("SELECT DISTINCT PHARMACY_ORDERABLE_ITEM FROM vista_pharmacy", conn)

# Prepare a set of uppercase drug names for efficient lookup
pharmacy_set = set(pharmacy_drugs['PHARMACY_ORDERABLE_ITEM'].str.upper().unique())

matched_count = 0
not_found_count = 0

# TODO: For each extraction, check if each medication name matches a pharmacy drug
# Use case-insensitive matching (e.g., compare .upper() versions)
# Print a summary: "TAMOXIFEN — FOUND in pharmacy" or "CISPLATIN — NOT FOUND"
for extraction in extractions:
    print(f"\nChecking medications for Patient: {extraction.patient_name}")

    for drug in extraction.medications:
        drug_name_upper = drug.name.upper().strip()

        if drug_name_upper in pharmacy_set:
            print(f"✅ {drug_name_upper} — FOUND in pharmacy")
            matched_count += 1
        else:
            print(f"❌ {drug_name_upper} — NOT FOUND")
            not_found_count += 1

# TODO: Report totals: X drugs matched, Y drugs not found
print("\n" + "="*30)
print(f"FINAL REPORT")
print(f"Drugs matched: {matched_count}")
print(f"Drugs not found: {not_found_count}")
print("="*30)



Checking medications for Patient: Layla Al-Masri
❌ DOXORUBICIN — NOT FOUND
❌ CYCLOPHOSPHAMIDE — NOT FOUND
❌ PACLITAXEL — NOT FOUND
✅ TAMOXIFEN — FOUND in pharmacy

Checking medications for Patient: Omar Khalil Zarqa
✅ METOCLOPRAMIDE — FOUND in pharmacy
✅ PARACETAMOL — FOUND in pharmacy
✅ ENOXAPARIN — FOUND in pharmacy

Checking medications for Patient: Hana Bisharat

Checking medications for Patient: Tariq Awad
✅ FILGRASTIM — FOUND in pharmacy
❌ FLUCONAZOLE — NOT FOUND
❌ ACYCLOVIR — NOT FOUND

Checking medications for Patient: Yazan Ahmad Tamimi
❌ 6-MERCAPTOPURINE — NOT FOUND
❌ METHOTREXATE — NOT FOUND
❌ VINCRISTINE — NOT FOUND
✅ DEXAMETHASONE — FOUND in pharmacy

Checking medications for Patient: Reem Sami Al-Khatib
❌ CISPLATIN — NOT FOUND
❌ ETOPOSIDE — NOT FOUND

Checking medications for Patient: Samer Fakhouri
❌ DOXORUBICIN — NOT FOUND
❌ BLEOMYCIN — NOT FOUND
❌ VINBLASTINE — NOT FOUND
❌ DACARBAZINE — NOT FOUND
✅ ONDANSETRON — FOUND in pharmacy
✅ DEXAMETHASONE — FOUND in pharmacy

C

### Cell 2.5 — Compare Against Ground Truth

Each note has an `expected_extraction` field with ground truth for `medications`, `allergies`,
and `potential_interactions`. Compare your LLM extractions against these:
- How many medications did you correctly identify?
- Do the allergies match?
- How many drug-drug interaction pairs did the LLM find vs the ground truth?

In [80]:
# === CELL 2.5: COMPARE AGAINST GROUND TRUTH ===

# TODO: For each note, compare:
#   - Number of medications extracted vs expected
#   - Whether allergies match
#   - Number of interaction pairs matched vs expected
#     Hint: normalize drug pairs with sorted() and .upper() for comparison
#   - Print a per-note accuracy summary

# TODO: Print overall summary:
#   - Total medications extracted vs total expected
#   - Total interactions extracted vs total expected
#   - Notes where extraction was complete vs partial

total_meds_expected = 0
total_meds_extracted = 0
total_meds_correct = 0

total_allergies_expected = 0
total_allergies_extracted = 0
total_allergies_correct = 0

total_interactions_expected = 0
total_interactions_extracted = 0
total_interactions_correct = 0

complete_matches = 0

for i, note_data in enumerate(notes):
    expected = note_data['expected_extraction']
    extracted = extractions[i]

    print(f"\n--- Comparing Note ID: {note_data['note_id']} (Patient: {extracted.patient_name}) ---")

    # --- Compare Medications ---
    # Fix: Changed m['drug_name'] to m['name'] to match the ground truth structure
    expected_meds_raw = [m['name'] for m in expected.get('medications', [])]
    extracted_meds_raw = [m.name for m in extracted.medications]

    expected_meds_set = set(m.upper().strip() for m in expected_meds_raw)
    extracted_meds_set = set(m.upper().strip() for m in extracted_meds_raw)

    correct_meds = len(expected_meds_set.intersection(extracted_meds_set))
    meds_match_status = "✅ MATCH" if expected_meds_set == extracted_meds_set else "❌ PARTIAL/MISMATCH"

    total_meds_expected += len(expected_meds_set)
    total_meds_extracted += len(extracted_meds_set)
    total_meds_correct += correct_meds

    print(f"  Medications: Expected {len(expected_meds_set)}, Extracted {len(extracted_meds_set)}, Correct {correct_meds} - {meds_match_status}")
    if expected_meds_set != extracted_meds_set:
        print(f"    Missing: {expected_meds_set - extracted_meds_set}")
        print(f"    Extra: {extracted_meds_set - expected_meds_set}")

    # --- Compare Allergies ---
    expected_allergies_raw = expected.get('allergies', [])
    extracted_allergies_raw = extracted.allergies

    expected_allergies_set = set(a.upper().strip() for a in expected_allergies_raw)
    extracted_allergies_set = set(a.upper().strip() for a in extracted_allergies_raw)

    allergies_match_status = "✅ MATCH" if expected_allergies_set == extracted_allergies_set else "❌ MISMATCH"
    correct_allergies = len(expected_allergies_set.intersection(extracted_allergies_set))

    total_allergies_expected += len(expected_allergies_set)
    total_allergies_extracted += len(extracted_allergies_set)
    total_allergies_correct += correct_allergies

    print(f"  Allergies: Expected {len(expected_allergies_set)}, Extracted {len(extracted_allergies_set)} - {allergies_match_status}")
    if expected_allergies_set != extracted_allergies_set:
        print(f"    Missing: {expected_allergies_set - extracted_allergies_set}")
        print(f"    Extra: {extracted_allergies_set - expected_allergies_set}")

    # --- Compare Potential Interactions ---
    # Normalize interaction pairs for comparison (order-independent)
    def normalize_interaction_pair(d1, d2):
        return frozenset({d1.upper().strip(), d2.upper().strip()})

    expected_interactions_pairs = set()
    for p in expected.get('potential_interactions', []):
        expected_interactions_pairs.add(normalize_interaction_pair(p['drug1'], p['drug2']))

    extracted_interactions_pairs = set()
    for p in extracted.potential_interactions:
        extracted_interactions_pairs.add(normalize_interaction_pair(p.drug1, p.drug2))

    correct_interactions = len(expected_interactions_pairs.intersection(extracted_interactions_pairs))
    interactions_match_status = "✅ MATCH" if expected_interactions_pairs == extracted_interactions_pairs else "❌ PARTIAL/MISMATCH"

    total_interactions_expected += len(expected_interactions_pairs)
    total_interactions_extracted += len(extracted_interactions_pairs)
    total_interactions_correct += correct_interactions

    print(f"  Interactions: Expected {len(expected_interactions_pairs)}, Extracted {len(extracted_interactions_pairs)}, Correct {correct_interactions} - {interactions_match_status}")
    if expected_interactions_pairs != extracted_interactions_pairs:
        print(f"    Missing: {expected_interactions_pairs - extracted_interactions_pairs}")
        print(f"    Extra: {extracted_interactions_pairs - expected_interactions_pairs}")

    if meds_match_status == "✅ MATCH" and allergies_match_status == "✅ MATCH" and interactions_match_status == "✅ MATCH":
        complete_matches += 1

print("\n" + "="*50)
print("OVERALL SUMMARY OF EXTRACTION ACCURACY")
print("="*50)

print(f"Total Notes Processed: {len(notes)}")
print(f"Complete Extractions (all categories matched perfectly): {complete_matches}")

print("\n--- Medications ---")
print(f"Total Expected: {total_meds_expected}")
print(f"Total Extracted: {total_meds_extracted}")
print(f"Total Correctly Identified: {total_meds_correct}")
print(f"Medication Precision: {total_meds_correct / total_meds_extracted:.2f}" if total_meds_extracted > 0 else "Precision: N/A")
print(f"Medication Recall: {total_meds_correct / total_meds_expected:.2f}" if total_meds_expected > 0 else "Recall: N/A")

print("\n--- Allergies ---")
print(f"Total Expected: {total_allergies_expected}")
print(f"Total Extracted: {total_allergies_extracted}")
print(f"Total Correctly Identified: {total_allergies_correct}")
print(f"Allergy Accuracy (Exact Set Match): {total_allergies_correct / total_allergies_expected:.2f}" if total_allergies_expected > 0 else "Accuracy: N/A")

print("\n--- Potential Interactions ---")
print(f"Total Expected: {total_interactions_expected}")
print(f"Total Extracted: {total_interactions_extracted}")
print(f"Total Correctly Identified: {total_interactions_correct}")
print(f"Interaction Precision: {total_interactions_correct / total_interactions_extracted:.2f}" if total_interactions_extracted > 0 else "Precision: N/A")
print(f"Interaction Recall: {total_interactions_correct / total_interactions_expected:.2f}" if total_interactions_expected > 0 else "Recall: N/A")
print("="*50)


--- Comparing Note ID: NOTE_001 (Patient: Layla Al-Masri) ---
  Medications: Expected 4, Extracted 4, Correct 4 - ✅ MATCH
  Allergies: Expected 1, Extracted 0 - ❌ MISMATCH
    Missing: {'NKDA'}
    Extra: set()
  Interactions: Expected 3, Extracted 0, Correct 0 - ❌ PARTIAL/MISMATCH
    Missing: {frozenset({'CYCLOPHOSPHAMIDE', 'DOXORUBICIN'}), frozenset({'PACLITAXEL', 'DOXORUBICIN'}), frozenset({'TAMOXIFEN', 'DOXORUBICIN'})}
    Extra: set()

--- Comparing Note ID: NOTE_002 (Patient: Omar Khalil Zarqa) ---
  Medications: Expected 3, Extracted 3, Correct 3 - ✅ MATCH
  Allergies: Expected 2, Extracted 2 - ✅ MATCH
  Interactions: Expected 1, Extracted 0, Correct 0 - ❌ PARTIAL/MISMATCH
    Missing: {frozenset({'PARACETAMOL', 'METOCLOPRAMIDE'})}
    Extra: set()

--- Comparing Note ID: NOTE_003 (Patient: Hana Bisharat) ---
  Medications: Expected 0, Extracted 0, Correct 0 - ✅ MATCH
  Allergies: Expected 0, Extracted 0 - ✅ MATCH
  Interactions: Expected 0, Extracted 0, Correct 0 - ✅ MATCH

-

---
# Part 3: Text-to-SQL — Pharmacist Natural Language Interface (25%)

Build a system where a pharmacist can ask questions in plain English
and get answers from the pharmacy database — no SQL knowledge required.

### Cell 3.1 — Schema Description

Create a detailed schema description string for the LLM. Include:
- Both table names and key column names with types
- Sample values for important columns
- The JOIN relationship (MRN)
- Data quirks (many NULL columns, STATUS values, ROUTE values, date formats)

In [81]:
# === CELL 3.1: SCHEMA DESCRIPTION ===

# TODO: Create a schema_description string that describes both tables
# Include the key pharmacy columns, patient columns, JOIN key, and data notes

schema_description = """
# Database Schema for KHCC Pharmacy System

## Table: vista_pharmacy
- **Description:** Contains individual medication orders for patients.
- **Key Columns & Types:**
    - `MRN` (TEXT/INTEGER): Medical Record Number, links to `vista_patients`.
    - `DRUG` (TEXT): Name of the prescribed medication. e.g., 'LETROZOLE 2.5MG TAB', 'ANASTROZOLE 1MG TAB'
    - `QTY` (INTEGER): Quantity of the medication dispensed.
    - `DAYS_SUPPLY` (INTEGER): Number of days the medication supply is intended for.
    - `Number_OF_REFILLS` (INTEGER): Number of refills remaining or originally authorized.
    - `ROUTE` (TEXT): Method of administration. e.g., 'ORAL', 'TOPICAL', 'SUBCUTANEOUS'
    - `SCHEDULE` (TEXT): Dosing schedule.
    - `STATUS` (TEXT): Current status of the order. e.g., 'EXPIRED', 'DISCONTINUED', 'ACTIVE', 'DISCONTINUED (EDIT)'
    - `CLINIC` (TEXT): Clinic where the prescription was ordered. e.g., 'PULMONARY-FERAS HAWARI', 'KHCC-ICU-CRITICAL CARE'
    - `PROVIDER` (TEXT): Prescribing physician.
    - `PHARMACY_ORDERABLE_ITEM` (TEXT): Standardized item name, useful for cross-referencing.
    - `DOSAGE_ORDERED` (TEXT): Specific dosage instructions.
    - `ISSUE_DATE` (TEXT/DATETIME): Date the prescription was issued.
    - `FILL_DATE` (TEXT/DATETIME): Date the prescription was filled.
    - `EXPIRATION_DATE` (TEXT/DATETIME): Date the prescription expires.
- **Data Notes:**
    - Many columns might contain NULL values.
    - Date columns (`ISSUE_DATE`, `FILL_DATE`, `EXPIRATION_DATE`) are stored as TEXT and should be converted to DATETIME for date-based operations (e.g., using `julianday()`).

## Table: vista_patients
- **Description:** Contains demographic information for patients.
- **Key Columns & Types:**
    - `MRN` (TEXT/INTEGER): Medical Record Number, links to `vista_pharmacy`.
    - `DOB` (TEXT/DATETIME): Date of Birth. Stored as TEXT.
    - `SEX` (TEXT): Patient's biological sex. e.g., 'FEMALE', 'MALE'
    - `NATIONALITY` (TEXT): Patient's nationality. e.g., 'JOR', 'PSE', 'SYR'
    - `PROVINCE` (TEXT): Patient's province/state. (Note: No 'CITY' column directly available).
- **Data Notes:**
    - `DOB` is stored as TEXT and should be parsed as a date for age calculations (e.g., using `julianday()`).
    - Available address-related columns are `ADDRESS_1`, `ADDRESS_2`, `ADDRESS_3`, `PROVINCE`, and `ZIP_CODE`.

## Relationship:
- Both tables can be joined on the `MRN` column (`vista_pharmacy.MRN = vista_patients.MRN`).

## Important Notes for SQL Generation:
- Prioritize `SELECT` statements. Avoid `INSERT`, `UPDATE`, `DELETE`, or `DROP`.
- Ensure all table and column names used in the query exist in the provided schema.
- Always alias table names (e.g., 'vista_pharmacy vp', 'vista_patients tp').
- When calculating age, use `(julianday('now') - julianday(tp.DOB)) / 365.25` for approximate years.
- Be mindful of case sensitivity in string comparisons (consider using `UPPER()` or `LOWER()` if necessary, though SQLite often defaults to case-insensitive for `LIKE`).
- When filtering by `CLINIC` use values as seen in `vista_pharmacy` (e.g., 'PULMONARY-FERAS HAWARI', 'KHCC-ICU-CRITICAL CARE').

"""

# TODO: Get sample rows from both tables to verify your schema
print("Sample from vista_pharmacy:")
print(pd.read_sql("SELECT DRUG, QTY, DAYS_SUPPLY, ROUTE, STATUS, CLINIC FROM vista_pharmacy LIMIT 3", conn))

print("\nSample from vista_patients:")
print(pd.read_sql("SELECT MRN, DOB, SEX, NATIONALITY FROM vista_patients LIMIT 3", conn))

Sample from vista_pharmacy:
                  DRUG  QTY  DAYS_SUPPLY         ROUTE   STATUS  \
0   TAMOXIFEN 20MG TAB   30           30          ORAL  EXPIRED   
1  ENOXAPARIN 40MG INJ   30           30  SUBCUTANEOUS  EXPIRED   
2  ANASTROZOLE 1MG TAB   30           30          ORAL  EXPIRED   

                   CLINIC  
0      BREAST ONC-GHADEER  
1  PULMONARY-FERAS HAWARI  
2     PEDIATRIC ONC-AHMAD  

Sample from vista_patients:
              MRN                            DOB     SEX NATIONALITY
0  21968531796627  1954-01-24T00:00:00.000+00:00    MALE         SYR
1  14391759658629  1948-06-01T00:00:00.000+00:00    MALE         JOR
2  19786466377132  1960-03-14T00:00:00.000+00:00  FEMALE         LBN


### Cell 3.2 — Text-to-SQL Function

Write a function `text_to_sql(question)` that sends the schema + question
to GPT and gets back a SQL query. Include:
- Safety rules (SELECT only)
- Instruction to return only the SQL, no explanation

In [82]:
# === CELL 3.2: TEXT TO SQL FUNCTION ===

# TODO: Write a function text_to_sql(question) that:
# 1. Builds a system prompt with the schema and safety rules
# 2. Sends the question to GPT-4o-mini
# 3. Returns the generated SQL string

def text_to_sql(question, few_shot_examples: list = None):
    system_prompt = f"""
You are an experienced clinical pharmacist who can translate natural language questions into SQLite SQL queries for data analysis.
Use the following database schema to generate your queries:

{schema_description}

Strictly follow these rules:
- Only generate SQL SELECT statements. Do NOT generate INSERT, UPDATE, DELETE, DROP, or any other DDL/DML statements.
- Ensure all table and column names used in the query exist in the provided schema.
- Always alias table names (e.g., 'vista_pharmacy vp', 'vista_patients tp').
- Do NOT include any explanations, comments, or additional text in your response, ONLY the SQL query.
- If a question cannot be answered from the provided schema, return an empty string or a SQL query that makes sense.
- When joining tables, always use the 'MRN' column for linking 'vista_pharmacy' and 'vista_patients'.
- For age calculations, use `(julianday('now') - julianday(tp.DOB)) / 365.25` for the `vista_patients` table.
"""

    messages = [{"role": "system", "content": system_prompt}]
    if few_shot_examples:
        messages.extend(few_shot_examples)
    messages.append({"role": "user", "content": question})

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            temperature=0.0,
        )
        sql_query = response.choices[0].message.content.strip()
        # Remove markdown code block delimiters if present
        if sql_query.startswith("```sql") and sql_query.endswith("```"):
            sql_query = sql_query[len("```sql"): -len("```")].strip()

        # Basic validation: ensure it's a SELECT statement and doesn't contain forbidden keywords
        if sql_query.upper().startswith("SELECT") and \
           all(keyword not in sql_query.upper() for keyword in ["INSERT", "UPDATE", "DELETE", "DROP", "CREATE", "ALTER"]):
            return sql_query
        else:
            print(f"Warning: Generated SQL query failed safety check: {sql_query}")
            return ""
    except Exception as e:
        print(f"Error generating SQL: {e}")
        return ""

# TODO: Test with: "How many total pharmacy orders are in the database?"
print("Test Query: How many total pharmacy orders are in the database?")
test_question = "How many total pharmacy orders are in the database?"
generated_sql = text_to_sql(test_question)
print(f"Generated SQL:\n{generated_sql}")

Test Query: How many total pharmacy orders are in the database?
Generated SQL:
SELECT COUNT(*) AS total_orders FROM vista_pharmacy vp;


### Cell 3.3 — Full Pipeline with Few-Shot Examples

Build `ask_pharmacy(question)` that:
1. Generates SQL (using few-shot examples for better accuracy)
2. Validates it is a SELECT query
3. Executes it against the database
4. Sends results back to GPT for a natural language answer

Include at least **3 few-shot examples** in your system prompt, including one JOIN example.

In [83]:
# === CELL 3.3: FULL PIPELINE WITH FEW-SHOT ===

# TODO: Write ask_pharmacy(question) that:
# 1. Calls text_to_sql() with few-shot examples
# 2. Validates it's a SELECT query (block dangerous keywords)
# 3. Executes the SQL with pd.read_sql()
# 4. Sends results to GPT for a natural language answer
# 5. Returns dict with: sql, answer, data

def ask_pharmacy(question):
    # Few-shot examples for SQL generation
    few_shot_examples_sql = [
        {"role": "user", "content": "What are the top 5 most prescribed drugs?"},
        {"role": "assistant", "content": "SELECT DRUG, COUNT(*) AS order_count FROM vista_pharmacy GROUP BY DRUG ORDER BY order_count DESC LIMIT 5"},
        {"role": "user", "content": "How many male vs female patients are there?"},
        {"role": "assistant", "content": "SELECT SEX, COUNT(DISTINCT MRN) AS patient_count FROM vista_patients GROUP BY SEX"},
        {"role": "user", "content": "List the medications for patients from Jordan over 60 years old."},
        {"role": "assistant", "content": "SELECT DISTINCT vp.DRUG FROM vista_pharmacy vp JOIN vista_patients tp ON vp.MRN = tp.MRN WHERE tp.NATIONALITY = 'JOR' AND (julianday('now') - julianday(tp.DOB)) / 365.25 > 60"}
    ]

    # Call text_to_sql function with the question and few-shot examples
    generated_sql = text_to_sql(question, few_shot_examples_sql)

    # The safety checks are now handled within text_to_sql
    if not generated_sql:
        return {"sql": "", "answer": "Could not generate a safe SQL query.", "data": pd.DataFrame()}

    # Execute the generated SQL query
    result_df = pd.DataFrame()
    try:
        result_df = pd.read_sql(generated_sql, conn)
    except Exception as e:
        print(f"Error executing SQL: {e}")
        return {"sql": generated_sql, "answer": f"Error executing SQL: {e}", "data": pd.DataFrame()}

    # Generate natural language answer from the SQL results
    system_prompt_answer = """
You are a helpful clinical informatician that summarizes SQL query results in a clear and concise natural language format for a pharmacist.
Present the key findings and answer the user's original question based ONLY on the provided query results.
If the results are empty, state that no relevant data was found.
"""
    # Convert DataFrame to a string for the LLM
    results_str = result_df.to_markdown(index=False)

    messages_answer = [
        {"role": "system", "content": system_prompt_answer},
        {"role": "user", "content": f"Original question: {question}\n\nSQL Query Results:\n{results_str}"}
    ]

    nl_answer = ""
    try:
        response_answer = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages_answer,
            temperature=0.0,
        )
        nl_answer = response_answer.choices[0].message.content.strip()
    except Exception as e:
        print(f"Error generating natural language answer: {e}")
        nl_answer = f"Error summarizing results: {e}"

    return {"sql": generated_sql, "answer": nl_answer, "data": result_df}

### Cell 3.4 — Test with 5 Pharmacy Questions

Run your pipeline on these 5 questions and print the SQL + answer for each:
1. "Which patients are on Tamoxifen?"
2. "What is the most prescribed drug in the ICU?"
3. "Show active prescriptions for patients from Amman"
4. "How many oral vs injectable medications are there?"
5. "Find patients with more than 5 pharmacy orders"

In [84]:
# === CELL 3.4: TEST QUESTIONS ===

questions = [
    "Which patients are on Tamoxifen?",
    "What is the most prescribed drug in the ICU?",
    "Show active prescriptions for patients from Amman",
    "How many oral vs injectable medications are there?",
    "Find patients with more than 5 pharmacy orders"
]

# TODO: Loop through questions, call ask_pharmacy(), print SQL + answer
for q in questions:
    result = ask_pharmacy(q)
    print(f"Q: {q}")
    print(f"SQL: {result['sql']}")
    print(f"Answer: {result['answer']}")
    print()

Q: Which patients are on Tamoxifen?
SQL: SELECT DISTINCT tp.MRN FROM vista_pharmacy vp JOIN vista_patients tp ON vp.MRN = tp.MRN WHERE vp.DRUG LIKE '%TAMOXIFEN%'
Answer: The query results indicate that there are 20 patients currently on Tamoxifen, identified by their medical record numbers (MRNs). If you need further details about these patients, please let me know.

Q: What is the most prescribed drug in the ICU?
SQL: SELECT DRUG, COUNT(*) AS order_count FROM vista_pharmacy WHERE CLINIC = 'KHCC-ICU-CRITICAL CARE' GROUP BY DRUG ORDER BY order_count DESC LIMIT 1
Answer: The most prescribed drug in the ICU is Gabapentin 300mg capsule, with a total of 6 orders.

Q: Show active prescriptions for patients from Amman
SQL: SELECT vp.* FROM vista_pharmacy vp JOIN vista_patients tp ON vp.MRN = tp.MRN WHERE vp.STATUS = 'ACTIVE' AND tp.PROVINCE = 'AMMAN'
Answer: The query results show a total of 50 active prescriptions for patients from Amman. Here are the key findings:

1. **Patient Demographics

---
# Part 4: Critical Analysis (15%)

Write **200-400 words** addressing all four questions below. This must reference YOUR specific implementation — generic answers will lose marks.

**Replace the TODOs below with your analysis.**

### 1. Drug Interaction Blind Spots
What can the LLM miss compared to a real clinical decision support system? Think about dose-dependent interactions, patient-specific contraindications (renal function, age), and temporal factors.

There is a clear issue with drug interactions even in this implmenetation, we could improve that by asking it to rersearch the known drug interactions, or by providing it with a knowledge base. These issues however, will be endless, and not ALL can be solved by adding knowledge. There will be always missing information, and there will always be a need for human in the loop.

### 2. Text-to-SQL Risks
What happens if the generated SQL is wrong but looks plausible? How would you validate results in a clinical pharmacy setting before acting on them?

There are mutiple ways we can validate the results, the most simple being human in the loop that checks if the results make sense first. There are also technical methods that can help, for example adding validation steps to the pipeline, possibly by a different model.


### 3. PHI / Security
If this pipeline were deployed at KHCC, what are the privacy and security risks? Consider: API key management, patient data sent to OpenAI, local regulations, and data residency.

There are many issues that need to be managed in a clinical setting. One is patient data privacy, this is essential and should be protected, for example by local deployment, or by deidentifying the patient data before it is given to the LLM according to guidlines. Some regulations in some countries have more strict data residency plocies, e.g. Saudi Arabia. Depending on the type of regulation, this is handled either by local deployment, or in-country deployment, i.e. the data should not leave the country.

### 4. Missing Data
Many pharmacy columns are NULL in our synthetic data. Which missing columns would a real system need? (Think: NDC codes, allergy cross-checking, insurance/billing, drug-drug interaction databases.)

This depends on the context we are applying this system for, but usually, larger scope data is needed. This is done to prevent any missing context. Take for example imaging data, if no mention of their results are present in the notes that we extracted in this code, it can cause in accuracies. As mentioned, allergy cross-checking is also important, as a mistake in such areas, especially due to misssing data, can be critical to patient's outcome.

---
# Part 5: Stretch Challenge (10%)

Choose **ONE** of the following options.

### Option A: Tool Calling Pharmacist Assistant

Define 2 tools:
- `run_sql_query(sql)` — executes a SELECT query and returns results
- `check_drug_interaction(drug1, drug2)` — returns interaction info

Build a multi-turn assistant that uses both tools to answer questions like:
"Is there an interaction between Tamoxifen and Anastrozole for patient MRN 123?"

### Option B: Text-to-SQL Benchmarking

Create **10 question / expected-SQL pairs**. Run each question through both:
- Zero-shot `text_to_sql()` (no examples)
- Few-shot `text_to_sql()` (with examples)

Compare accuracy: which questions improved with few-shot? Which did not? Report your findings.

In [85]:
# === PART 5: STRETCH CHALLENGE ===
# TODO: Implement your chosen option (A or B)


---
## Submission Checklist

- [ ] Part 1: Pharmacy data loaded into SQLite with 5 SQL queries and 3 JOIN queries
- [ ] Part 2: Pydantic models defined, all 10 notes extracted, interactions compared to ground truth, cross-referenced with pharmacy DB
- [ ] Part 3: text_to_sql and ask_pharmacy functions working, 5 test questions answered
- [ ] Part 4: Critical analysis (200-400 words) referencing your implementation
- [ ] Part 5: Stretch challenge completed (Option A or B)
- [ ] Notebook runs top-to-bottom without errors
- [ ] Pushed to GitHub with 3+ descriptive commits
- [ ] No real patient data in the notebook